In [1]:
import pandas as pd
import numpy as np

# -----------------------------
# 1) LOAD
# -----------------------------
hmda = pd.read_csv("hmda_sample.csv", low_memory=False)
crosswalk = pd.read_csv("tract_zip_sample.csv", low_memory=False)

In [2]:
# -----------------------------
# 2) CLEAN HMDA TRACT GEOID
# -----------------------------
hmda["state_fips_clean"] = (
    pd.to_numeric(hmda["state_code"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(2)
)

hmda["county_fips_clean"] = (
    pd.to_numeric(hmda["county_code"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(3)
)

hmda["tract_code_clean"] = (
    np.round(pd.to_numeric(hmda["census_tract_number"], errors="coerce") * 100)
    .astype("Int64")
    .astype(str)
    .str.zfill(6)
)

hmda["tract_geoid"] = (
    hmda["state_fips_clean"]
    + hmda["county_fips_clean"]
    + hmda["tract_code_clean"]
)

# -----------------------------
# 3) CLEAN CROSSWALK
# -----------------------------
crosswalk["tract_geoid"] = (
    pd.to_numeric(crosswalk["tract"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(11)
)

crosswalk["zip_clean"] = (
    pd.to_numeric(crosswalk["zip"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

for col in ["res_ratio", "bus_ratio", "oth_ratio", "tot_ratio"]:
    crosswalk[col] = pd.to_numeric(crosswalk[col], errors="coerce")

# keep only usable rows
crosswalk = crosswalk.loc[
    crosswalk["tract_geoid"].notna()
    & crosswalk["zip_clean"].notna()
    & crosswalk["tot_ratio"].notna()
].copy()

# optional: check tract weights sum to ~1
tract_weight_check = (
    crosswalk.groupby("tract_geoid", as_index=False)["tot_ratio"]
    .sum()
    .rename(columns={"tot_ratio": "tot_ratio_sum"})
)

print(tract_weight_check["tot_ratio_sum"].describe())

count    57099.000000
mean         0.761979
std          0.353386
min          0.000072
25%          0.549114
50%          0.999453
75%          1.000000
max          1.000000
Name: tot_ratio_sum, dtype: float64


In [3]:
# -----------------------------
# 4) MANY-TO-MANY WEIGHTED JOIN
# -----------------------------
hmda_weighted = hmda.merge(
    crosswalk[["tract_geoid", "zip_clean", "tot_ratio", "res_ratio", "bus_ratio", "oth_ratio"]],
    on="tract_geoid",
    how="left"
)

# -----------------------------
# 5) ALLOCATE COUNTS / AMOUNTS
# -----------------------------
hmda_weighted["loan_count_alloc"] = hmda_weighted["tot_ratio"]

hmda_weighted["loan_amount_000s_num"] = pd.to_numeric(
    hmda_weighted["loan_amount_000s"], errors="coerce"
)
hmda_weighted["applicant_income_000s_num"] = pd.to_numeric(
    hmda_weighted["applicant_income_000s"], errors="coerce"
)

hmda_weighted["loan_amount_alloc_000s"] = (
    hmda_weighted["loan_amount_000s_num"] * hmda_weighted["tot_ratio"]
)

hmda_weighted["applicant_income_alloc_000s"] = (
    hmda_weighted["applicant_income_000s_num"] * hmda_weighted["tot_ratio"]
)

# -----------------------------
# 6) EXAMPLE: ZIP-LEVEL AGGREGATION
# -----------------------------
zip_summary = (
    hmda_weighted.groupby("zip_clean", dropna=False)
    .agg(
        allocated_loan_count=("loan_count_alloc", "sum"),
        allocated_loan_amount_000s=("loan_amount_alloc_000s", "sum"),
        allocated_applicant_income_000s=("applicant_income_alloc_000s", "sum"),
    )
    .reset_index()
)

# weighted averages at ZIP level
zip_summary["avg_loan_amount_000s"] = (
    zip_summary["allocated_loan_amount_000s"] / zip_summary["allocated_loan_count"]
)

zip_summary["avg_applicant_income_000s"] = (
    zip_summary["allocated_applicant_income_000s"] / zip_summary["allocated_loan_count"]
)

In [4]:
zip_summary.head()

,zip_clean,allocated_loan_count,allocated_loan_amount_000s,allocated_applicant_income_000s,avg_loan_amount_000s,avg_applicant_income_000s
0,00602,1.000000,149.000000,96.000000,149.000000,96.000000
1,00603,0.154157,29.130082,5.535875,188.964095,35.910698
2,00606,0.002691,0.279892,0.104960,104.000000,39.000000
3,00612,0.007722,0.833977,0.378378,108.000000,49.000000
4,00623,2.999761,367.985192,146.994268,122.671497,49.001990


In [ ]:
#For zip-level summaries:
hmda_weighted["loan_amount_000s_num"] = pd.to_numeric(hmda_weighted["loan_amount_000s"], errors="coerce")
hmda_weighted["loan_amount_alloc_000s"] = hmda_weighted["loan_amount_000s_num"] * hmda_weighted["tot_ratio"]

zip_by_action = (
    hmda_weighted.groupby(["zip_clean", "action_taken_name"], dropna=False)
    .agg(
        allocated_loan_count=("tot_ratio", "sum"),
        allocated_loan_amount_000s=("loan_amount_alloc_000s", "sum"),
    )
    .reset_index()
)